# 1. 도서 제목 및 도서 설명 정의

In [1]:
doc5_titles = [
    "자바스크립트언어",
    "일본관광시기",
    "파이썬언어",
    "기계학습기초",
    "스페인방문계절"
]

doc5_desc = [
    "자바스크립트는 웹 개발에 필수적인 프로그래밍 언어입니다.",
    "일본은 벚꽃이 피는 봄이 관광하기 가장 좋은 시기입니다.",
    "파이썬 언어는 데이터분석과 기계학습에 효율적인 프로그래밍 언어입니다",
    "기계학습은 데이터를 활용하여 컴퓨터가 학습하도록 하는 기술입니다.",
    "스페인은 날씨가 온화한 봄이나 가을에 방문하는 것이 이상적입니다."
]

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="",
    base_url="https://gms.ssafy.io/gmsapi/api.openai.com/v1"
)

response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = doc5_desc
).data

embedding_vectors = [i.embedding for i in response]

In [3]:
# 임베딩 벡터 데이터만 리스트로 추출
embedding_vectors = [i.embedding for i in response]
# 생성된 벡터 데이터수 == 문서 수
print(len(embedding_vectors))
# 업스테이지 임베딩 모델의 벡터 차원 == 4096
print(len(embedding_vectors[0]))

5
1536


# 2. 문서 간 코싸인 유사도 계산

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(embedding_vectors, embedding_vectors)
print(similarities)

[[1.         0.09787161 0.36122694 0.25013722 0.16311998]
 [0.09787161 1.         0.05174922 0.03216215 0.4898752 ]
 [0.36122694 0.05174922 1.         0.45043242 0.10764046]
 [0.25013722 0.03216215 0.45043242 1.         0.07790908]
 [0.16311998 0.4898752  0.10764046 0.07790908 1.        ]]


# 3. 첫번째 책의 유사도 정렬

In [8]:
# 첫번째 책의 각 책과의 유사도 정보 (index, 유사도)
sim_scores = list(enumerate(similarities[0]))

# 유사 기준 내림차순 정렬
sim_scores.sort(key = lambda x: x[1], reverse = True)

# 자기 자신은 제외한 상위 3개 문서 출력
print(sim_scores[1:4] )

[(2, np.float64(0.3612269357679635)), (3, np.float64(0.25013721917537507)), (4, np.float64(0.1631199802043586))]


# 4. 책 정보가 유사한 도서 추천

In [9]:
books_titles = doc5_titles

def recommendations(title):
    # 책의 제목을 입력하면 해당 제목의 인덱스를 리턴받아 idx에 저장.
    if title in books_titles :
        idx=books_titles.index(title)
        similar_doc = similarities[idx]
    else :
        print("도서 정보가 존재하지 않습니다.")
        return []

    # 입력된 책과 줄거리(document embedding)가 유사한 책 3개 선정.
    sim_scores = list(enumerate(similar_doc)) # (index, 유사도) 튜플의 리스트
    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True) # 유사도 높은 순서로 sorting
    sim_scores = sim_scores[1:4] # 자기 자신은 제외한 상위 3개
    print(sim_scores)

    similar_books_titles = []
    # 유사한 책 제목 출력
    for index, book_info in sim_scores:
        title=books_titles[index]
        similar_books_titles.append(title)

    return similar_books_titles


In [10]:

title="소년이 온다"
#title = "이반 일리치의 죽음"
recommendations(title)

도서 정보가 존재하지 않습니다.


[]

In [11]:
title="자바스크립트언어"
# title="스페인방문계절"
recommendations(title)

[(2, np.float64(0.3612269357679635)), (3, np.float64(0.25013721917537507)), (4, np.float64(0.1631199802043586))]


['파이썬언어', '기계학습기초', '스페인방문계절']